# 04 — cross-stage synthesis

**triplet-phase-lock**

Pipeline:
- Pi → expand (structure)
- π → extend (local drift)
- Π → resist (threshold filtering)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8,4)
plt.rcParams['axes.grid'] = True

In [ ]:
def sequence_n(k):
    return 24.0*k - 25.0

def sequence_n_perturbed(k, amplitude=8.0, period=6.0):
    return sequence_n(k) + amplitude*np.sin(k/period)

def sequence_n_noisy(k, amplitude=40.0, seed=7):
    rng = np.random.default_rng(seed)
    return sequence_n(k) + rng.normal(0, amplitude, len(k))

def build_triplets(x):
    return np.stack([x[:-2], x[1:-1], x[2:]], axis=1)

def normalize_rows(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n,1e-12)

def cosine_scores(x, ref):
    return normalize_rows(x) @ (ref/np.linalg.norm(ref))

def direction_change(x):
    d = np.diff(x,axis=0)
    dn = normalize_rows(d)
    return np.linalg.norm(dn[1:] - dn[:-1], axis=1)

def local_delta(x):
    return np.linalg.norm(np.diff(x,axis=0), axis=1)

In [ ]:
k = np.arange(1,83)

families = {
    'clean': sequence_n(k),
    'weak': sequence_n_perturbed(k,8,6),
    'strong': sequence_n_perturbed(k,20,4),
    'noisy': sequence_n_noisy(k,40)
}

trip = {n: build_triplets(v) for n,v in families.items()}

## Expand (Pi)

In [ ]:
plt.figure()
for n,v in families.items():
    plt.plot(k,v,label=n)
plt.title('Expand: structure invariant')
plt.legend()
plt.show()

## Extend (π)

In [ ]:
delta = {n: local_delta(t) for n,t in trip.items()}
drift = {n: direction_change(t) for n,t in trip.items()}

In [ ]:
plt.figure()
for n in trip:
    plt.plot(delta[n],label=n)
plt.title('Extend: local magnitude')
plt.legend()
plt.show()

In [ ]:
plt.figure()
for n in trip:
    plt.plot(drift[n],label=n)
plt.title('Extend: directional change')
plt.legend()
plt.show()

## Resist (Π)

In [ ]:
ref = normalize_rows(trip['clean']).mean(axis=0)
ref = ref/np.linalg.norm(ref)

scores = {n: cosine_scores(t,ref) for n,t in trip.items()}

thr45 = 1/np.sqrt(2)
thrS = 0.9995

maskS = {n: scores[n] >= thrS for n in trip}

In [ ]:
plt.figure()
for n in trip:
    plt.plot(scores[n],label=n)
plt.axhline(thr45,ls='--')
plt.axhline(thrS,ls='--')
plt.title('Resist: cosine scores')
plt.legend()
plt.show()

In [ ]:
plt.figure()
for n in trip:
    plt.plot(maskS[n].astype(float),label=n)
plt.title('Resist: strict threshold')
plt.legend()
plt.show()

## 🔥 Cross-stage link (NEW)

In [ ]:
plt.figure()
for n in trip:
    plt.scatter(drift[n], scores[n][1:], label=n)

plt.xlabel('directional change')
plt.ylabel('cosine score')
plt.title('Drift vs Resistance')
plt.legend()
plt.show()

## Summary

expand → invariant structure

extend → drift emerges

resist → threshold filters drift

Key:

drift ↑ ⇒ resistance ↓